# CacheBlendPlus: Full Colab Experiment (Mistral)

This notebook is a complete, reproducible experiment driver for your presentation/report.

It will:
1. Clone your GitHub repository (public or private)
2. Install dependencies
3. Load a Mistral-family model (4-bit by default for Colab viability)
4. Run 3-way TTFT comparisons:
   - Cold prefill
   - Warm standard KV cache
   - Warm CacheBlend
5. Sweep multiple recompute ratios
6. Save experiment outputs to JSON + CSV
7. Generate report-ready visualizations

## 0) Runtime Requirements

- In Colab: **Runtime -> Change runtime type -> T4 GPU**
- Recommended: Colab Pro for longer sessions

If your repository or model is private, set secrets in Colab:
- Left sidebar -> **Secrets** -> add:
  - `GITHUB_TOKEN` (GitHub fine-grained PAT with repo read access)
  - `HF_TOKEN` (Hugging Face token with model access)

You can still run without tokens for public repo + fully public model.

In [ ]:
# 1) Configure experiment

# REQUIRED: replace with your GitHub repo URL
REPO_HTTPS_URL = "https://github.com/<your_user>/<your_repo>.git"
REPO_DIR_NAME = "CacheBlendPlus"
# Branch you will keep updating from local machine
GIT_BRANCH = "feat/token-selector"

# Model options:
# - mistralai/Mistral-7B-Instruct-v0.2   (heavier)
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0  (faster fallback if OOM/time-limited)
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# Sequence/workload settings
MAX_CHUNK_LEN = 384
MAX_PROMPT_LEN = 128
TRIALS = 3
WARMUP_RUNS = 1
RATIO_SWEEP = [0.05, 0.10, 0.15, 0.20, 0.30]

# Output paths
RESULTS_DIR = "/content/results"
JSON_PATH = f"{RESULTS_DIR}/cacheblend_mistral_results.json"
CSV_PATH = f"{RESULTS_DIR}/cacheblend_mistral_results.csv"

print("Configured. Edit this cell before running if needed.")

In [ ]:
# 2) Read secrets and clone repo (public/private)
import os
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

repo_url = REPO_HTTPS_URL
if GITHUB_TOKEN and REPO_HTTPS_URL.startswith("https://"):
    # Token-auth URL for private repos
    repo_url = REPO_HTTPS_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if Path(REPO_DIR_NAME).exists():
    print(f"Repo directory '{REPO_DIR_NAME}' already exists; using existing checkout.")
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, repo_url, REPO_DIR_NAME], check=True)

%cd /content/{REPO_DIR_NAME}
# Ensure we are on the configured branch
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", GIT_BRANCH], check=True)
subprocess.run(["git", "pull", "origin", GIT_BRANCH], check=True)
print("Current working directory:", os.getcwd())
print("Active branch synced:", GIT_BRANCH)

In [ ]:
# 3b) Pull latest code from branch (run this anytime after pushing new commits)
import os
import subprocess

os.chdir(f"/content/{REPO_DIR_NAME}")

print(f"Syncing latest changes from origin/{GIT_BRANCH} ...")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", GIT_BRANCH], check=True)
subprocess.run(["git", "pull", "origin", GIT_BRANCH], check=True)

print("Sync complete.")
subprocess.run(["git", "status", "-sb"], check=True)

In [ ]:
# 3) Install dependencies
pip install -U pip
pip install "transformers>=4.45,<5" datasets evaluate rouge_score sentence-transformers bitsandbytes accelerate ninja pytest tqdm pandas matplotlib seaborn

In [ ]:
# 4) Imports + model load
import json
import time
from statistics import mean, pstdev

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from cacheblendplus.kv_store import KVCacheStore
from cacheblendplus.adaptive_selector import AdaptiveTokenSelector
from cacheblendplus.recompute_engine import SelectiveRecomputer
from cacheblendplus.pipeline import KVBlender, cacheblend_generate

assert torch.cuda.is_available(), "GPU runtime required. Set Colab runtime to T4 GPU."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN if HF_TOKEN else None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="eager",  # More tolerant for mixed cache dtypes in Colab runs
    token=HF_TOKEN if HF_TOKEN else None,
)
model.eval()

print("Model loaded.")

In [ ]:
# 5) Workload setup (balanced and fact-sensitive)
# This workload is still valid for exact-hit latency benchmarking,
# but the prompt is designed to depend on concrete facts.

chunk_texts = [
    (
        "Project Atlas v2 rollout notes: Phase Alpha starts in July with 12 partner teams. "
        "The memory budget target is 16GB and battery target is 72Wh for the reference device. "
        "Launch readiness requires passing integration gates A, B, and C. "
    ) * 4,
    (
        "Policy revision memo: reporting cadence moved from quarterly to semi-annual for standard teams. "
        "High-risk teams remain on quarterly reporting with additional audit checkpoints. "
        "Penalty framework changed from immediate fines to staged escalation. "
    ) * 4,
    (
        "City infrastructure update: Metro line M4 opening shifted from June to September. "
        "Station count increased from 18 to 22 and projected daily ridership increased by 15 percent. "
        "Funding allocation now prioritizes signaling reliability improvements. "
    ) * 4,
    (
        "Service SLO update: p95 latency target tightened from 220ms to 180ms for premium tier traffic. "
        "Error budget policy now enforces freeze windows after consecutive weekly breaches. "
        "Mitigation playbook emphasizes cache invalidation hygiene and rollout canaries. "
    ) * 4,
]

prompt = (
    "Summarize the latest factual updates: rollout month, reporting cadence, metro opening month, "
    "and premium-tier p95 target."
)

print("Chunk count:", len(chunk_texts))
print("Prompt:", prompt)
print("Approx chunk token lengths:", [len(tokenizer(c, truncation=True, max_length=MAX_CHUNK_LEN)["input_ids"]) for c in chunk_texts])

In [ ]:
# 6) Benchmark utilities (includes HKVD fusor mode)
from cacheblendplus.kv_store import pack_kv, unpack_kv
from cacheblendplus.token_selector import CacheBlendFusor


def make_components(k_ratio: float):
    store = KVCacheStore()
    selector = AdaptiveTokenSelector(
        model=model,
        # Keep adaptive behavior but centered around requested sweep ratio.
        min_k_ratio=max(0.01, k_ratio * 0.5),
        max_k_ratio=min(0.50, k_ratio * 1.5),
        low_thresh=0.20,
        high_thresh=1.10,
    )
    recomputer = SelectiveRecomputer(model)
    blender = KVBlender()
    return store, selector, recomputer, blender


def cacheblend_generate_hkvd(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    store,
    fusor,
    max_new_tokens=64,
    min_new_tokens=8,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    benchmark_first_token=False,
):
    cache_hits = 0
    cache_misses = 0
    fused_kv = None
    hkvd_stats_all = []

    torch.cuda.synchronize()
    t_request_start = time.perf_counter()

    for chunk_text in chunk_texts:
        chunk_enc = tokenizer(
            chunk_text,
            return_tensors="pt",
            return_attention_mask=True,
            truncation=True,
            max_length=MAX_CHUNK_LEN,
        )
        chunk_ids = chunk_enc["input_ids"].to("cuda")
        chunk_attn = chunk_enc["attention_mask"].to("cuda")

        cached = store.load(chunk_text, device="cuda")

        if cached is None:
            cache_misses += 1
            with torch.no_grad():
                out = model(chunk_ids, attention_mask=chunk_attn, use_cache=True)
            chunk_kv = pack_kv(out.past_key_values)
            store.store(chunk_text, chunk_kv)
        else:
            cache_hits += 1
            chunk_kv = cached
            hit_mask = torch.ones(chunk_ids.shape[1], device=chunk_ids.device, dtype=torch.bool)
            chunk_kv, hkvd_layers = fusor.fuse(chunk_ids, chunk_kv, hit_mask)
            hkvd_stats_all.append(fusor.get_stats(hkvd_layers, int(chunk_ids.shape[1])))

        fused_kv = chunk_kv if fused_kv is None else torch.cat([fused_kv, chunk_kv], dim=2)

    prompt_enc = tokenizer(
        prompt,
        return_tensors="pt",
        return_attention_mask=True,
        truncation=True,
        max_length=MAX_PROMPT_LEN,
    )
    prompt_ids = prompt_enc["input_ids"].to("cuda")
    prompt_attn = prompt_enc["attention_mask"].to("cuda")

    past = unpack_kv(fused_kv) if fused_kv is not None else None

    torch.cuda.synchronize()
    t_decode_start = time.perf_counter()

    context_len = fused_kv.shape[2] if fused_kv is not None else 0
    cache_position = torch.arange(
        context_len,
        context_len + prompt_ids.shape[1],
        device=prompt_ids.device,
    )

    gen_max = 1 if benchmark_first_token else max_new_tokens
    gen_min = 1 if benchmark_first_token else min_new_tokens
    gen_sample = False if benchmark_first_token else do_sample

    gen_kwargs = {
        "attention_mask": prompt_attn,
        "past_key_values": past,
        "cache_position": cache_position,
        "max_new_tokens": gen_max,
        "min_new_tokens": gen_min,
        "use_cache": True,
        "do_sample": gen_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if gen_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        out_ids = model.generate(prompt_ids, **gen_kwargs)

    torch.cuda.synchronize()
    generate_only_ms = (time.perf_counter() - t_decode_start) * 1000
    ttft_ms = (time.perf_counter() - t_request_start) * 1000

    generated = out_ids[0][prompt_ids.shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    hkvd_final_ratio_mean = None
    hkvd_savings_mean = None
    if hkvd_stats_all:
        hkvd_final_ratio_mean = float(np.mean([s["final_layer_ratio"] for s in hkvd_stats_all]))
        hkvd_savings_mean = float(np.mean([s["true_savings_pct"] for s in hkvd_stats_all]))

    return {
        "text": text,
        "ttft_ms": ttft_ms,
        "e2e_ttft_ms": ttft_ms,
        "generate_only_ms": generate_only_ms,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "hkvd_final_ratio_mean": hkvd_final_ratio_mean,
        "hkvd_savings_pct_mean": hkvd_savings_mean,
    }


def run_one_mode(mode: str, k_ratio: float):
    store, selector, recomputer, blender = make_components(k_ratio)
    fusor = CacheBlendFusor(model, r=float(k_ratio))

    # Prime cache to create warm-hit scenario for warm modes.
    _ = cacheblend_generate(
        prompt,
        chunk_texts,
        model,
        tokenizer,
        store,
        selector,
        recomputer,
        blender,
        mode="standard_cache",
        benchmark_first_token=True,
    )

    if mode == "cold":
        # Fresh store => cold prefill baseline
        store, selector, recomputer, blender = make_components(k_ratio)
        out = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store,
            selector,
            recomputer,
            blender,
            mode="standard_cache",
            benchmark_first_token=True,
        )
        stats = {}
    elif mode == "warm_standard":
        out = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store,
            selector,
            recomputer,
            blender,
            mode="standard_cache",
            benchmark_first_token=True,
        )
        stats = {}
    elif mode == "warm_cacheblend":
        out = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store,
            selector,
            recomputer,
            blender,
            mode="cacheblend",
            benchmark_first_token=True,
        )
        stats = selector.get_last_selection_stats() if hasattr(selector, "get_last_selection_stats") else {}
    elif mode == "warm_hkvd":
        out = cacheblend_generate_hkvd(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store,
            fusor,
            benchmark_first_token=True,
        )
        stats = {
            "hkvd_final_ratio_mean": out.get("hkvd_final_ratio_mean", None),
            "hkvd_savings_pct_mean": out.get("hkvd_savings_pct_mean", None),
        }
    else:
        raise ValueError(mode)

    return out, stats


def run_full_experiment(ratios, trials=3, warmup_runs=1):
    records = []

    # Warmup
    for _ in range(warmup_runs):
        _ = run_one_mode("warm_standard", ratios[0])

    for r in ratios:
        for t in range(1, trials + 1):
            cold_out, _ = run_one_mode("cold", r)
            std_out, _ = run_one_mode("warm_standard", r)
            cb_out, cb_stats = run_one_mode("warm_cacheblend", r)
            hkvd_out, hkvd_stats = run_one_mode("warm_hkvd", r)

            records.extend([
                {
                    "ratio": r,
                    "trial": t,
                    "mode": "cold",
                    "ttft_ms": cold_out["ttft_ms"],
                    "generate_only_ms": cold_out.get("generate_only_ms", None),
                    "cache_hits": cold_out.get("cache_hits", None),
                    "cache_misses": cold_out.get("cache_misses", None),
                },
                {
                    "ratio": r,
                    "trial": t,
                    "mode": "warm_standard",
                    "ttft_ms": std_out["ttft_ms"],
                    "generate_only_ms": std_out.get("generate_only_ms", None),
                    "cache_hits": std_out.get("cache_hits", None),
                    "cache_misses": std_out.get("cache_misses", None),
                },
                {
                    "ratio": r,
                    "trial": t,
                    "mode": "warm_cacheblend",
                    "ttft_ms": cb_out["ttft_ms"],
                    "generate_only_ms": cb_out.get("generate_only_ms", None),
                    "cache_hits": cb_out.get("cache_hits", None),
                    "cache_misses": cb_out.get("cache_misses", None),
                    "selected_k_ratio": cb_stats.get("selected_k_ratio", None),
                    "selected_k": cb_stats.get("selected_k", None),
                    "mean_divergence": cb_stats.get("mean_divergence", None),
                },
                {
                    "ratio": r,
                    "trial": t,
                    "mode": "warm_hkvd",
                    "ttft_ms": hkvd_out["ttft_ms"],
                    "generate_only_ms": hkvd_out.get("generate_only_ms", None),
                    "cache_hits": hkvd_out.get("cache_hits", None),
                    "cache_misses": hkvd_out.get("cache_misses", None),
                    "hkvd_final_ratio_mean": hkvd_stats.get("hkvd_final_ratio_mean", None),
                    "hkvd_savings_pct_mean": hkvd_stats.get("hkvd_savings_pct_mean", None),
                },
            ])

    return pd.DataFrame(records)

In [ ]:
# 7) Run experiment
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

df = run_full_experiment(RATIO_SWEEP, trials=TRIALS, warmup_runs=WARMUP_RUNS)

summary = (
    df.groupby(["ratio", "mode"], as_index=False)
      .agg(
          ttft_mean_ms=("ttft_ms", "mean"),
          ttft_std_ms=("ttft_ms", "std"),
          n=("ttft_ms", "count"),
      )
)

# Relative speedups vs cold (within each ratio)
wide = summary.pivot(index="ratio", columns="mode", values="ttft_mean_ms").reset_index()
if "cold" in wide and "warm_standard" in wide:
    wide["speedup_cold_vs_warm_standard"] = wide["cold"] / wide["warm_standard"]
if "cold" in wide and "warm_cacheblend" in wide:
    wide["speedup_cold_vs_warm_cacheblend"] = wide["cold"] / wide["warm_cacheblend"]
if "cold" in wide and "warm_hkvd" in wide:
    wide["speedup_cold_vs_warm_hkvd"] = wide["cold"] / wide["warm_hkvd"]

print("Summary:")
display(summary)
print("Speedup table:")
display(wide)

results = {
    "config": {
        "model_id": MODEL_ID,
        "trials": TRIALS,
        "warmup_runs": WARMUP_RUNS,
        "ratio_sweep": RATIO_SWEEP,
        "max_chunk_len": MAX_CHUNK_LEN,
        "max_prompt_len": MAX_PROMPT_LEN,
    },
    "summary": summary.to_dict(orient="records"),
    "speedup": wide.to_dict(orient="records"),
    "raw": df.to_dict(orient="records"),
}

with open(JSON_PATH, "w") as f:
    json.dump(results, f, indent=2)

df.to_csv(CSV_PATH, index=False)

print(f"Saved JSON -> {JSON_PATH}")
print(f"Saved CSV  -> {CSV_PATH}")

In [ ]:
# 8) Visualization 1: TTFT by mode and ratio
sns.set_theme(style="whitegrid", context="talk")

plt.figure(figsize=(12, 7))
ax = sns.lineplot(
    data=summary,
    x="ratio",
    y="ttft_mean_ms",
    hue="mode",
    marker="o",
)

# Add error bars manually from std
for _, row in summary.iterrows():
    if pd.notna(row["ttft_std_ms"]):
        ax.errorbar(
            row["ratio"],
            row["ttft_mean_ms"],
            yerr=row["ttft_std_ms"],
            fmt="none",
            capsize=4,
            alpha=0.7,
            color="black",
        )

plt.title("TTFT vs Recompute Ratio (Mistral)")
plt.xlabel("Target Recompute Ratio")
plt.ylabel("TTFT (ms)")
plt.legend(title="Mode")
plt.tight_layout()
plt.show()

In [ ]:
# 9) Visualization 2: Speedup bars by ratio
speed_cols = [
    c
    for c in [
        "speedup_cold_vs_warm_standard",
        "speedup_cold_vs_warm_cacheblend",
        "speedup_cold_vs_warm_hkvd",
    ]
    if c in wide.columns
]
plot_df = wide[["ratio"] + speed_cols].melt(id_vars="ratio", var_name="comparison", value_name="speedup")

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_df, x="ratio", y="speedup", hue="comparison")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.title("Cold-Start Speedup by Ratio")
plt.xlabel("Target Recompute Ratio")
plt.ylabel("Speedup (x)")
plt.legend(title="Comparison")
plt.tight_layout()
plt.show()

In [ ]:
# 10) Visualization 3: Adaptive selector behavior
cb_df = df[df["mode"] == "warm_cacheblend"].copy()
if "selected_k_ratio" in cb_df.columns and cb_df["selected_k_ratio"].notna().any():
    plt.figure(figsize=(12, 7))
    sns.boxplot(data=cb_df, x="ratio", y="selected_k_ratio")
    plt.title("Observed Adaptive k-ratio Distribution")
    plt.xlabel("Target Recompute Ratio")
    plt.ylabel("Observed selected_k_ratio")
    plt.tight_layout()
    plt.show()

if "mean_divergence" in cb_df.columns and cb_df["mean_divergence"].notna().any():
    plt.figure(figsize=(12, 7))
    sns.scatterplot(data=cb_df, x="mean_divergence", y="ttft_ms", hue="ratio", palette="viridis", s=100)
    plt.title("Divergence vs TTFT (CacheBlend Warm Mode)")
    plt.xlabel("Mean divergence")
    plt.ylabel("TTFT (ms)")
    plt.tight_layout()
    plt.show()

hkvd_df = df[df["mode"] == "warm_hkvd"].copy()
if "hkvd_final_ratio_mean" in hkvd_df.columns and hkvd_df["hkvd_final_ratio_mean"].notna().any():
    plt.figure(figsize=(12, 7))
    sns.boxplot(data=hkvd_df, x="ratio", y="hkvd_final_ratio_mean")
    plt.title("HKVD Final-Layer Ratio Distribution")
    plt.xlabel("Target r")
    plt.ylabel("Observed final_layer_ratio")
    plt.tight_layout()
    plt.show()

if "hkvd_savings_pct_mean" in hkvd_df.columns and hkvd_df["hkvd_savings_pct_mean"].notna().any():
    plt.figure(figsize=(12, 7))
    sns.scatterplot(data=hkvd_df, x="hkvd_savings_pct_mean", y="ttft_ms", hue="ratio", palette="magma", s=100)
    plt.title("HKVD Estimated Savings vs TTFT")
    plt.xlabel("Estimated true_savings_pct")
    plt.ylabel("TTFT (ms)")
    plt.tight_layout()
    plt.show()

## Notes for Presentation and Report

- Use the `summary` table and the three plots as your primary visuals.
- Recommended claims format:
  - "Warm standard cache achieved X.x speedup over cold prefill."
  - "CacheBlend warm achieved Y.y speedup at ratio r=..."
  - "Observed adaptive ratio trends indicate ..."
- If `warm_cacheblend` is slower than `warm_standard`, frame it honestly as:
  - correctness and reproducibility achieved,
  - optimization opportunity in recompute/blend overhead,
  - next work: CUDA kernel tuning and selector threshold calibration.

## Common Issues

1. **OOM on Mistral 7B**
   - Reduce chunk size/repetitions.
   - Lower trials.
   - Switch model to `TinyLlama/TinyLlama-1.1B-Chat-v1.0` for quick dry runs.

2. **Private model/repo auth failures**
   - Ensure `HF_TOKEN` and `GITHUB_TOKEN` are set in Colab Secrets.

3. **Slow runtime**
   - Lower `TRIALS` first.
   - Use smaller workload for iteration, then re-run full workload once.

## 11) Stale-Context Quality vs Latency Track

This section demonstrates the report narrative:
- `warm_standard` is usually fastest,
- but can lose quality when cache is stale/mismatched,
- while `warm_cacheblend` may recover quality by selective recomputation.

Method:
1. Build a cache from `cached_chunks`.
2. Reuse that cache for `query_chunks` (edited/updated context) by mapping stale KV to query keys.
3. Compare modes against a cold reference output from the updated context.

Quality metric: **ROUGE-L** vs cold reference output (proxy for faithfulness to updated context).

In [ ]:
# 12) Stale-context experiment setup
import evaluate

QUALITY_RATIO = 0.15
QUALITY_TRIALS = 2
QUALITY_MAX_NEW_TOKENS = 64
QUALITY_MIN_NEW_TOKENS = 24

rouge = evaluate.load("rouge")

stale_cases = [
    {
        "case_id": "capital_update",
        "prompt": "What is the capital of Kazakhstan today?",
        "cached_chunks": [
            "Kazakhstan's capital is Astana. It was briefly renamed to Nur-Sultan in 2019.",
            "In 2022, the city name reverted from Nur-Sultan back to Astana.",
        ],
        "query_chunks": [
            "Kazakhstan's capital is Astana. The 2019 rename to Nur-Sultan was later reversed.",
            "As of recent updates, the official capital name is Astana.",
        ],
    },
    {
        "case_id": "policy_revision",
        "prompt": "Summarize the latest policy status.",
        "cached_chunks": [
            "The policy draft stated mandatory quarterly reporting for all teams.",
            "Enforcement was planned to begin in Q1 with strict penalties.",
        ],
        "query_chunks": [
            "The revised policy now requires semi-annual reporting for most teams.",
            "Enforcement start was postponed, and penalties were softened in the update.",
        ],
    },
    {
        "case_id": "product_specs_change",
        "prompt": "What are the current product specs and launch timing?",
        "cached_chunks": [
            "The product launch was planned for June with 8GB memory and a 60Wh battery.",
            "Initial batch targeted enterprise customers only.",
        ],
        "query_chunks": [
            "The updated plan moves launch to September with 16GB memory and a 72Wh battery.",
            "Rollout now includes enterprise and education segments.",
        ],
    },
]

print("Stale cases:", [c["case_id"] for c in stale_cases])

In [ ]:
# 13) Run stale-context quality/latency benchmark + visualizations

def build_stale_mapped_store(cached_chunks, query_chunks):
    """
    Build KV from cached_chunks, but store under query_chunks keys to simulate stale cache reuse.

    Important: selector/recompute path expects cached KV token length to match query token length.
    We align stale KV length to query length by truncating or zero-padding along token dimension.
    """
    store = KVCacheStore()
    assert len(cached_chunks) == len(query_chunks)

    for cached_text, query_text in zip(cached_chunks, query_chunks):
        # Length target comes from the query side (what pipeline will run on).
        q_enc = tokenizer(
            query_text,
            return_tensors="pt",
            return_attention_mask=True,
            truncation=True,
            max_length=MAX_CHUNK_LEN,
        )
        q_len = int(q_enc["input_ids"].shape[1])

        c_enc = tokenizer(
            cached_text,
            return_tensors="pt",
            return_attention_mask=True,
            truncation=True,
            max_length=MAX_CHUNK_LEN,
        )
        c_ids = c_enc["input_ids"].to("cuda")
        c_attn = c_enc["attention_mask"].to("cuda")

        with torch.no_grad():
            out = model(c_ids, attention_mask=c_attn, use_cache=True)
        kv = pack_kv(out.past_key_values)  # (L, 2, N_cached, H, D)

        n_cached = int(kv.shape[2])
        if n_cached > q_len:
            kv = kv[:, :, :q_len, :, :]
        elif n_cached < q_len:
            pad = torch.zeros(
                kv.shape[0], kv.shape[1], q_len - n_cached, kv.shape[3], kv.shape[4],
                dtype=kv.dtype,
                device=kv.device,
            )
            kv = torch.cat([kv, pad], dim=2)

        # Store under query text key to force a warm hit with stale KV.
        store.store(query_text, kv)

    return store


def make_runtime_components(k_ratio):
    selector = AdaptiveTokenSelector(
        model=model,
        min_k_ratio=max(0.01, k_ratio * 0.5),
        max_k_ratio=min(0.50, k_ratio * 1.5),
        low_thresh=0.20,
        high_thresh=1.10,
    )
    recomputer = SelectiveRecomputer(model)
    blender = KVBlender()
    return selector, recomputer, blender


quality_records = []
example_outputs = []

for case in stale_cases:
    for trial in range(1, QUALITY_TRIALS + 1):
        case_id = case["case_id"]
        prompt_q = case["prompt"]
        query_chunks = case["query_chunks"]
        cached_chunks = case["cached_chunks"]

        # Cold reference on updated context
        cold_store = KVCacheStore()
        sel_c, rec_c, bl_c = make_runtime_components(QUALITY_RATIO)
        cold_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            cold_store,
            sel_c,
            rec_c,
            bl_c,
            mode="standard_cache",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        # Warm standard on stale cache
        stale_store_std = build_stale_mapped_store(cached_chunks, query_chunks)
        sel_s, rec_s, bl_s = make_runtime_components(QUALITY_RATIO)
        warm_std_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            stale_store_std,
            sel_s,
            rec_s,
            bl_s,
            mode="standard_cache",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        # Warm CacheBlend on stale cache
        stale_store_cb = build_stale_mapped_store(cached_chunks, query_chunks)
        sel_cb, rec_cb, bl_cb = make_runtime_components(QUALITY_RATIO)
        warm_cb_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            stale_store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        ref_text = cold_out["text"]
        std_text = warm_std_out["text"]
        cb_text = warm_cb_out["text"]

        std_rouge = rouge.compute(predictions=[std_text], references=[ref_text])["rougeL"]
        cb_rouge = rouge.compute(predictions=[cb_text], references=[ref_text])["rougeL"]

        quality_records.extend([
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "cold_reference",
                "ttft_ms": cold_out["ttft_ms"],
                "rougeL_vs_cold": 1.0,
            },
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "warm_standard_stale",
                "ttft_ms": warm_std_out["ttft_ms"],
                "rougeL_vs_cold": std_rouge,
            },
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "warm_cacheblend_stale",
                "ttft_ms": warm_cb_out["ttft_ms"],
                "rougeL_vs_cold": cb_rouge,
                "selected_k_ratio": sel_cb.get_last_selection_stats().get("selected_k_ratio", None),
                "mean_divergence": sel_cb.get_last_selection_stats().get("mean_divergence", None),
            },
        ])

        if trial == 1:
            example_outputs.append(
                {
                    "case_id": case_id,
                    "prompt": prompt_q,
                    "cold_reference": ref_text,
                    "warm_standard_stale": std_text,
                    "warm_cacheblend_stale": cb_text,
                }
            )

quality_df = pd.DataFrame(quality_records)
quality_summary = (
    quality_df.groupby("mode", as_index=False)
    .agg(
        ttft_mean_ms=("ttft_ms", "mean"),
        ttft_std_ms=("ttft_ms", "std"),
        rougeL_mean=("rougeL_vs_cold", "mean"),
        rougeL_std=("rougeL_vs_cold", "std"),
        n=("mode", "count"),
    )
)

print("Quality summary:")
display(quality_summary)

print("Sample qualitative outputs (trial 1 per case):")
display(pd.DataFrame(example_outputs))

# Plot A: Latency comparison in stale context
plt.figure(figsize=(12, 7))
sns.barplot(data=quality_summary, x="mode", y="ttft_mean_ms")
plt.title("Stale-Context Latency by Mode")
plt.xlabel("Mode")
plt.ylabel("TTFT (ms)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

# Plot B: Quality comparison in stale context
plt.figure(figsize=(12, 7))
sns.barplot(data=quality_summary, x="mode", y="rougeL_mean")
plt.title("Stale-Context Quality (ROUGE-L vs Cold Reference)")
plt.xlabel("Mode")
plt.ylabel("ROUGE-L")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

# Plot C: Quality-latency frontier
plt.figure(figsize=(12, 7))
sns.scatterplot(data=quality_summary, x="ttft_mean_ms", y="rougeL_mean", hue="mode", s=200)
for _, row in quality_summary.iterrows():
    plt.text(row["ttft_mean_ms"], row["rougeL_mean"], row["mode"], fontsize=10)
plt.title("Quality-Latency Frontier (Stale Context)")
plt.xlabel("TTFT mean (ms)")
plt.ylabel("ROUGE-L mean")
plt.tight_layout()
plt.show()

# Save stale-track outputs
quality_json_path = f"{RESULTS_DIR}/cacheblend_stale_quality_results.json"
quality_csv_path = f"{RESULTS_DIR}/cacheblend_stale_quality_results.csv"

with open(quality_json_path, "w") as f:
    json.dump(
        {
            "config": {
                "quality_ratio": QUALITY_RATIO,
                "quality_trials": QUALITY_TRIALS,
                "max_new_tokens": QUALITY_MAX_NEW_TOKENS,
                "min_new_tokens": QUALITY_MIN_NEW_TOKENS,
            },
            "summary": quality_summary.to_dict(orient="records"),
            "raw": quality_df.to_dict(orient="records"),
            "examples": example_outputs,
        },
        f,
        indent=2,
    )

quality_df.to_csv(quality_csv_path, index=False)
print(f"Saved stale-quality JSON -> {quality_json_path}")
print(f"Saved stale-quality CSV  -> {quality_csv_path}")

## 13b) Enhanced Stale-Context Benchmark (adds HKVD + factual scoring)

Run this cell instead of the older stale benchmark cell when you want:
- `warm_hkvd_stale` included
- ROUGE-L vs cold reference
- simple factual keyword match score (case-specific)

This gives a stronger report table for quality/latency tradeoffs.

In [ ]:
# 13c) Run enhanced stale benchmark (standard vs cacheblend vs HKVD)
import re

# Case-specific factual keywords for a lightweight factuality score.
# Score = fraction of keywords present in generated output.
FACT_KEYWORDS = {
    "capital_update": ["astana", "capital"],
    "policy_revision": ["semi-annual", "quarterly", "postponed", "softened"],
    "product_specs_change": ["september", "16gb", "72wh", "education"],
}


def normalize_text(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def factual_keyword_score(text: str, keywords):
    t = normalize_text(text)
    if not keywords:
        return 0.0
    hits = sum(1 for k in keywords if k.lower() in t)
    return hits / len(keywords)


def make_hkvd_fusor(k_ratio: float):
    # CacheBlendFusor imported earlier in benchmark utilities cell.
    return CacheBlendFusor(model, r=float(k_ratio))


enhanced_records = []
enhanced_examples = []

for case in stale_cases:
    case_id = case["case_id"]
    prompt_q = case["prompt"]
    query_chunks = case["query_chunks"]
    cached_chunks = case["cached_chunks"]
    keywords = FACT_KEYWORDS.get(case_id, [])

    for trial in range(1, QUALITY_TRIALS + 1):
        # 1) Cold reference on updated context
        cold_store = KVCacheStore()
        sel_c, rec_c, bl_c = make_runtime_components(QUALITY_RATIO)
        cold_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            cold_store,
            sel_c,
            rec_c,
            bl_c,
            mode="standard_cache",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        # 2) Warm standard with stale cache
        stale_store_std = build_stale_mapped_store(cached_chunks, query_chunks)
        sel_s, rec_s, bl_s = make_runtime_components(QUALITY_RATIO)
        warm_std_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            stale_store_std,
            sel_s,
            rec_s,
            bl_s,
            mode="standard_cache",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        # 3) Warm modular CacheBlend with stale cache
        stale_store_cb = build_stale_mapped_store(cached_chunks, query_chunks)
        sel_cb, rec_cb, bl_cb = make_runtime_components(QUALITY_RATIO)
        warm_cb_out = cacheblend_generate(
            prompt_q,
            query_chunks,
            model,
            tokenizer,
            stale_store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=False,
            do_sample=False,
            max_new_tokens=QUALITY_MAX_NEW_TOKENS,
            min_new_tokens=QUALITY_MIN_NEW_TOKENS,
        )

        # 4) Warm HKVD fusor with stale cache
        stale_store_hkvd = build_stale_mapped_store(cached_chunks, query_chunks)
        fusor = make_hkvd_fusor(QUALITY_RATIO)
        hkvd_error = None
        try:
            warm_hkvd_out = cacheblend_generate_hkvd(
                prompt_q,
                query_chunks,
                model,
                tokenizer,
                stale_store_hkvd,
                fusor,
                benchmark_first_token=False,
                do_sample=False,
                max_new_tokens=QUALITY_MAX_NEW_TOKENS,
                min_new_tokens=QUALITY_MIN_NEW_TOKENS,
            )
        except Exception as e:
            hkvd_error = str(e)
            warm_hkvd_out = {
                "text": "",
                "ttft_ms": float("nan"),
                "cache_hits": None,
                "cache_misses": None,
                "hkvd_final_ratio_mean": None,
                "hkvd_savings_pct_mean": None,
            }

        # Metrics against cold reference text
        ref_text = cold_out["text"]
        std_text = warm_std_out["text"]
        cb_text = warm_cb_out["text"]
        hkvd_text = warm_hkvd_out["text"]

        std_rouge = rouge.compute(predictions=[std_text], references=[ref_text])["rougeL"]
        cb_rouge = rouge.compute(predictions=[cb_text], references=[ref_text])["rougeL"]
        hkvd_rouge = rouge.compute(predictions=[hkvd_text], references=[ref_text])["rougeL"] if hkvd_error is None else float("nan")

        cold_fact = factual_keyword_score(ref_text, keywords)
        std_fact = factual_keyword_score(std_text, keywords)
        cb_fact = factual_keyword_score(cb_text, keywords)
        hkvd_fact = factual_keyword_score(hkvd_text, keywords) if hkvd_error is None else float("nan")

        enhanced_records.extend([
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "cold_reference",
                "ttft_ms": cold_out["ttft_ms"],
                "rougeL_vs_cold": 1.0,
                "fact_score": cold_fact,
            },
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "warm_standard_stale",
                "ttft_ms": warm_std_out["ttft_ms"],
                "rougeL_vs_cold": std_rouge,
                "fact_score": std_fact,
            },
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "warm_cacheblend_stale",
                "ttft_ms": warm_cb_out["ttft_ms"],
                "rougeL_vs_cold": cb_rouge,
                "fact_score": cb_fact,
                "selected_k_ratio": sel_cb.get_last_selection_stats().get("selected_k_ratio", None),
                "mean_divergence": sel_cb.get_last_selection_stats().get("mean_divergence", None),
            },
            {
                "case_id": case_id,
                "trial": trial,
                "mode": "warm_hkvd_stale",
                "ttft_ms": warm_hkvd_out["ttft_ms"],
                "rougeL_vs_cold": hkvd_rouge,
                "fact_score": hkvd_fact,
                "hkvd_final_ratio_mean": warm_hkvd_out.get("hkvd_final_ratio_mean", None),
                "hkvd_savings_pct_mean": warm_hkvd_out.get("hkvd_savings_pct_mean", None),
                "hkvd_error": hkvd_error,
            },
        ])

        if trial == 1:
            enhanced_examples.append(
                {
                    "case_id": case_id,
                    "prompt": prompt_q,
                    "cold_reference": ref_text,
                    "warm_standard_stale": std_text,
                    "warm_cacheblend_stale": cb_text,
                    "warm_hkvd_stale": hkvd_text,
                    "hkvd_error": hkvd_error,
                }
            )

enhanced_df = pd.DataFrame(enhanced_records)
enhanced_summary = (
    enhanced_df.groupby("mode", as_index=False)
    .agg(
        ttft_mean_ms=("ttft_ms", "mean"),
        ttft_std_ms=("ttft_ms", "std"),
        rougeL_mean=("rougeL_vs_cold", "mean"),
        rougeL_std=("rougeL_vs_cold", "std"),
        fact_mean=("fact_score", "mean"),
        fact_std=("fact_score", "std"),
        n=("mode", "count"),
    )
)

print("Enhanced quality summary:")
display(enhanced_summary)

print("Enhanced sample outputs (trial 1 per case):")
display(pd.DataFrame(enhanced_examples))

# Plot 1: latency
plt.figure(figsize=(12, 7))
sns.barplot(data=enhanced_summary, x="mode", y="ttft_mean_ms")
plt.title("Stale-Context Latency by Mode (Enhanced)")
plt.xlabel("Mode")
plt.ylabel("TTFT (ms)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Plot 2: ROUGE-L
plt.figure(figsize=(12, 7))
sns.barplot(data=enhanced_summary, x="mode", y="rougeL_mean")
plt.title("Stale-Context ROUGE-L vs Cold Reference (Enhanced)")
plt.xlabel("Mode")
plt.ylabel("ROUGE-L")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Plot 3: Factual keyword score
plt.figure(figsize=(12, 7))
sns.barplot(data=enhanced_summary, x="mode", y="fact_mean")
plt.title("Stale-Context Factual Keyword Score (Enhanced)")
plt.xlabel("Mode")
plt.ylabel("Factual score (0-1)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Plot 4: quality-latency frontier with factual color
plt.figure(figsize=(12, 7))
sns.scatterplot(data=enhanced_summary, x="ttft_mean_ms", y="rougeL_mean", hue="fact_mean", style="mode", s=220)
for _, row in enhanced_summary.iterrows():
    plt.text(row["ttft_mean_ms"], row["rougeL_mean"], row["mode"], fontsize=10)
plt.title("Quality-Latency Frontier (Enhanced Stale Context)")
plt.xlabel("TTFT mean (ms)")
plt.ylabel("ROUGE-L mean")
plt.tight_layout()
plt.show()

# Save outputs
enhanced_json_path = f"{RESULTS_DIR}/cacheblend_stale_quality_enhanced.json"
enhanced_csv_path = f"{RESULTS_DIR}/cacheblend_stale_quality_enhanced.csv"

with open(enhanced_json_path, "w") as f:
    json.dump(
        {
            "config": {
                "quality_ratio": QUALITY_RATIO,
                "quality_trials": QUALITY_TRIALS,
                "max_new_tokens": QUALITY_MAX_NEW_TOKENS,
                "min_new_tokens": QUALITY_MIN_NEW_TOKENS,
                "fact_keywords": FACT_KEYWORDS,
            },
            "summary": enhanced_summary.to_dict(orient="records"),
            "raw": enhanced_df.to_dict(orient="records"),
            "examples": enhanced_examples,
        },
        f,
        indent=2,
    )

enhanced_df.to_csv(enhanced_csv_path, index=False)
print(f"Saved enhanced stale-quality JSON -> {enhanced_json_path}")
print(f"Saved enhanced stale-quality CSV  -> {enhanced_csv_path}")

## 14) Real Dataset Experiment — HotpotQA (Multi-Hop RAG)

**Why HotpotQA?**
- Closest available proxy to the paper's 2WikiMQA benchmark (both are multi-hop Wikipedia QA)
- Each question requires reading **2 supporting passages** → naturally tests cross-attention benefit
- Gold answers provided → ROUGE-L vs ground truth (more rigorous than vs cold output)
- Ships with context paragraphs bundled → no external retriever needed

**Correct experimental framing (read before running):**
- Primary latency claim: `warm_cacheblend` vs `cold` (CacheBlend is faster than full prefill)
- Primary quality claim: `warm_cacheblend` ROUGE-L vs `warm_standard` ROUGE-L (CacheBlend preserves cross-attention)
- `warm_standard` is ALWAYS faster than `warm_cacheblend` on latency — that is expected and correct

**Setup:**
1. Load 30 HotpotQA validation examples
2. For each question: extract its 2 supporting passages as chunks
3. Run query twice: cold (first call, cache empty) then warm (second call, cache populated)
4. Measure TTFT for cold vs warm_cacheblend vs warm_standard
5. Measure ROUGE-L vs gold answer for quality

In [ ]:
# 14a) Load HotpotQA and build per-query chunk lists

from datasets import load_dataset
import hashlib, re

# HotpotQA "distractor" split: each sample has 10 context paragraphs
# but only 2 are the true supporting facts. We use only those 2.
# Load 30 samples — enough for meaningful stats without OOM on T4.
HQ_N_SAMPLES = 30
HQ_MAX_CHUNK_TOKENS = 256   # shorter than MAX_CHUNK_LEN to stay under VRAM budget
HQ_MAX_PROMPT_TOKENS = 96
HQ_MAX_NEW_TOKENS = 48
HQ_MIN_NEW_TOKENS = 8
HQ_TARGET_RATIO = 0.15      # CacheBlend recompute ratio for this experiment

print("Loading HotpotQA validation split...")
hq_dataset = load_dataset(
    "hotpot_qa",
    "distractor",
    split=f"validation[:{HQ_N_SAMPLES}]",
    trust_remote_code=True,
)
print(f"Loaded {len(hq_dataset)} examples.")

def get_supporting_chunks(sample, max_chunks=2):
    """
    Return text of the supporting paragraphs for this question.
    Uses supporting_facts titles to identify which context paragraphs are relevant.
    Falls back to first 2 context paragraphs if supporting_facts lookup fails.
    """
    sp_titles = set(fact[0] for fact in sample["supporting_facts"])
    chunks = []
    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in sp_titles:
            text = title.strip() + ": " + " ".join(s.strip() for s in sentences)
            chunks.append(text)
        if len(chunks) >= max_chunks:
            break
    # Fallback: if lookup missed, take first 2
    if len(chunks) == 0:
        for title, sentences in zip(
            sample["context"]["title"][:max_chunks],
            sample["context"]["sentences"][:max_chunks],
        ):
            text = title.strip() + ": " + " ".join(s.strip() for s in sentences)
            chunks.append(text)
    return chunks

def build_rag_prompt(question: str) -> str:
    return f"Answer the following question concisely based on the provided context.\nQuestion: {question}\nAnswer:"

# Pre-build per-sample (chunks, prompt, gold_answer)
hq_samples = []
for sample in hq_dataset:
    chunks = get_supporting_chunks(sample, max_chunks=2)
    if not chunks:
        continue
    hq_samples.append({
        "id": sample["id"],
        "question": sample["question"],
        "gold_answer": sample["answer"],
        "chunks": chunks,
        "prompt": build_rag_prompt(sample["question"]),
    })

print(f"Prepared {len(hq_samples)} samples with supporting chunks.")
print("\nExample:")
print("  Q:", hq_samples[0]["question"])
print("  A:", hq_samples[0]["gold_answer"])
print("  Chunk 1 (first 120 chars):", hq_samples[0]["chunks"][0][:120])
print("  Chunk count:", len(hq_samples[0]["chunks"]))
print("  Approx chunk token lengths:", [
    len(tokenizer(c, truncation=True, max_length=HQ_MAX_CHUNK_TOKENS)["input_ids"])
    for c in hq_samples[0]["chunks"]
])

In [ ]:
# 14b) Run HotpotQA benchmark — 4-way comparison
#
# Modes:
#   cold                    : fresh store, full prefill — baseline
#   warm_standard           : cache hit, zero recompute — fastest, may lose quality
#   warm_cacheblend_hkvd    : cache hit + paper-faithful HKVD fusor (fixed r=0.15)
#   warm_cacheblend_adaptive: cache hit + HKVD fusor with adaptive r (Extension C)
#                             r derived from layer-0 L2 KV deviation inside fuse()
#                             — SINGLE forward pass, same as hkvd mode
#
# PRIMARY LATENCY CLAIM  : warm_cacheblend_hkvd speedup vs cold
# EXTENSION C CLAIM      : warm_cacheblend_adaptive ROUGE-L >= warm_cacheblend_hkvd ROUGE-L
#                          at similar or lower latency (adaptive r < fixed r on easy chunks)

import gc
from cacheblendplus.token_selector import CacheBlendFusor

# ── Shared components ──────────────────────────────────────────────────────────
# selector kept for backward compat; not called in these modes
hq_selector   = AdaptiveTokenSelector(
    model=model,
    base_k_ratio=HQ_TARGET_RATIO,
    min_k_ratio=0.05,
    max_k_ratio=0.30,
    low_thresh=0.05,
    high_thresh=0.20,
)
# fusor: paper-faithful HKVD engine — used by both hkvd and adaptive modes
hq_fusor      = CacheBlendFusor(model, r=HQ_TARGET_RATIO)
hq_recomputer = SelectiveRecomputer(model)
hq_blender    = KVBlender()

# Global store — chunks cached across samples
hq_global_store = KVCacheStore()
hq_records = []

def run_hq_ttft(sample_dict, store, mode):
    out = cacheblend_generate(
        sample_dict["prompt"], sample_dict["chunks"],
        model, tokenizer, store,
        selector=hq_selector, recomputer=hq_recomputer,
        blender=hq_blender, fusor=hq_fusor,
        mode=mode, max_new_tokens=1, min_new_tokens=1,
        do_sample=False, benchmark_first_token=True,
    )
    return out["ttft_ms"], out["cache_hits"], out["cache_misses"]


def run_hq_quality(sample_dict, store, mode):
    out = cacheblend_generate(
        sample_dict["prompt"], sample_dict["chunks"],
        model, tokenizer, store,
        selector=hq_selector, recomputer=hq_recomputer,
        blender=hq_blender, fusor=hq_fusor,
        mode=mode,
        max_new_tokens=HQ_MAX_NEW_TOKENS, min_new_tokens=HQ_MIN_NEW_TOKENS,
        do_sample=False, benchmark_first_token=False,
    )
    return out["text"], out["ttft_ms"]


print(f"Running 4-way HotpotQA benchmark on {len(hq_samples)} samples...")
print("Modes: cold | warm_standard | warm_cacheblend_hkvd | warm_cacheblend_adaptive\n")

for idx, sample in enumerate(hq_samples):
    print(f"  [{idx+1}/{len(hq_samples)}] {sample['question'][:65]}...")

    # 1) COLD
    cold_store = KVCacheStore()
    cold_ttft, _, _ = run_hq_ttft(sample, cold_store, "standard_cache")

    # Prime global store
    run_hq_ttft(sample, hq_global_store, "standard_cache")

    # 2) WARM STANDARD
    std_ttft, std_hits, _ = run_hq_ttft(sample, hq_global_store, "standard_cache")

    # 3) WARM CACHEBLEND HKVD (fixed r)
    hkvd_ttft, hkvd_hits, _ = run_hq_ttft(sample, hq_global_store, "cacheblend_hkvd")
    hkvd_r_used = hq_fusor.last_r_eff   # always == HQ_TARGET_RATIO for fixed mode

    # 4) WARM CACHEBLEND ADAPTIVE (r from layer-0 L2 deviation, single pass)
    ada_ttft, ada_hits, _   = run_hq_ttft(sample, hq_global_store, "cacheblend_adaptive")
    ada_r_used = hq_fusor.last_r_eff    # effective r chosen by Extension C

    # Quality (full generation vs gold)
    std_text,  _ = run_hq_quality(sample, hq_global_store, "standard_cache")
    hkvd_text, _ = run_hq_quality(sample, hq_global_store, "cacheblend_hkvd")
    ada_text,  _ = run_hq_quality(sample, hq_global_store, "cacheblend_adaptive")

    gold = sample["gold_answer"]
    std_rouge  = rouge.compute(predictions=[std_text],  references=[gold])["rougeL"]
    hkvd_rouge = rouge.compute(predictions=[hkvd_text], references=[gold])["rougeL"]
    ada_rouge  = rouge.compute(predictions=[ada_text],  references=[gold])["rougeL"]

    hq_records.append({
        "sample_id"    : sample["id"],
        "question"     : sample["question"],
        "gold_answer"  : gold,
        # TTFT
        "cold_ttft_ms" : cold_ttft,
        "std_ttft_ms"  : std_ttft,
        "hkvd_ttft_ms" : hkvd_ttft,
        "ada_ttft_ms"  : ada_ttft,
        # Speedup vs cold
        "std_speedup"  : cold_ttft / std_ttft   if std_ttft  > 0 else float("nan"),
        "hkvd_speedup" : cold_ttft / hkvd_ttft  if hkvd_ttft > 0 else float("nan"),
        "ada_speedup"  : cold_ttft / ada_ttft   if ada_ttft  > 0 else float("nan"),
        # Adaptive r tracking (Extension C analysis)
        "hkvd_r_used"  : hkvd_r_used,
        "ada_r_used"   : ada_r_used,
        # Quality vs gold
        "std_rougeL"   : std_rouge,
        "hkvd_rougeL"  : hkvd_rouge,
        "ada_rougeL"   : ada_rouge,
        # Generated text
        "std_text"     : std_text,
        "hkvd_text"    : hkvd_text,
        "ada_text"     : ada_text,
    })

    torch.cuda.empty_cache()
    gc.collect()

hq_df = pd.DataFrame(hq_records)

# ── Summary ───────────────────────────────────────────────────────────────────
num_cols = ["cold_ttft_ms","std_ttft_ms","hkvd_ttft_ms","ada_ttft_ms",
            "std_speedup","hkvd_speedup","ada_speedup",
            "hkvd_r_used","ada_r_used",
            "std_rougeL","hkvd_rougeL","ada_rougeL"]
hq_summary = hq_df[num_cols].agg(["mean","std","median"]).round(4)

print("\n=== 4-Way HotpotQA Benchmark Summary ===")
display(hq_summary)
print(f"\nSpeedup vs cold — standard        : {hq_df['std_speedup'].mean():.2f}×")
print(f"Speedup vs cold — hkvd (r={HQ_TARGET_RATIO}) : {hq_df['hkvd_speedup'].mean():.2f}×")
print(f"Speedup vs cold — adaptive         : {hq_df['ada_speedup'].mean():.2f}×")
print(f"\nMean r used — hkvd (fixed)         : {hq_df['hkvd_r_used'].mean():.4f}")
print(f"Mean r used — adaptive (Extension C): {hq_df['ada_r_used'].mean():.4f}")
print(f"\nROUGE-L vs gold — standard         : {hq_df['std_rougeL'].mean():.4f}")
print(f"ROUGE-L vs gold — hkvd             : {hq_df['hkvd_rougeL'].mean():.4f}  Δ vs std:  {hq_df['hkvd_rougeL'].mean()-hq_df['std_rougeL'].mean():+.4f}")
print(f"ROUGE-L vs gold — adaptive         : {hq_df['ada_rougeL'].mean():.4f}  Δ vs hkvd: {hq_df['ada_rougeL'].mean()-hq_df['hkvd_rougeL'].mean():+.4f}")

# Save
hq_json_path = f"{RESULTS_DIR}/hq_4way_results.json"
hq_csv_path  = f"{RESULTS_DIR}/hq_4way_results.csv"
hq_df.to_csv(hq_csv_path, index=False)
with open(hq_json_path, "w") as f:
    json.dump({
        "config": {
            "dataset": "hotpot_qa/distractor",
            "n_samples": len(hq_samples),
            "target_ratio": HQ_TARGET_RATIO,
            "max_chunk_tokens": HQ_MAX_CHUNK_TOKENS,
            "max_new_tokens": HQ_MAX_NEW_TOKENS,
            "modes": ["cold","warm_standard","warm_cacheblend_hkvd","warm_cacheblend_adaptive"],
            "adaptive_note": "r derived from layer-0 L2 KV deviation via x/(x+1) normalization",
        },
        "summary": hq_summary.to_dict(),
        "raw": hq_df.drop(columns=["std_text","hkvd_text","ada_text"]).to_dict(orient="records"),
    }, f, indent=2)
print(f"\nSaved → {hq_json_path}")
print(f"Saved → {hq_csv_path}")

In [ ]:
# 14b-ext) Extension C analysis — adaptive r distribution
# Shows whether Extension C is actually varying r or clamping to min/max.
# Key insight: if mean ada_r << hkvd_r, Extension C saves compute on easy chunks.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Extension C: Adaptive r Distribution Analysis", fontsize=14)

# Plot 1: r distribution histogram
ax1 = axes[0]
ax1.hist(hq_df["ada_r_used"], bins=15, color="#ffc000", alpha=0.8, edgecolor="black")
ax1.axvline(HQ_TARGET_RATIO, color="green", linestyle="--", linewidth=2, label=f"Fixed r={HQ_TARGET_RATIO}")
ax1.axvline(hq_df["ada_r_used"].mean(), color="red", linestyle="-", linewidth=2,
            label=f"Mean adaptive r={hq_df['ada_r_used'].mean():.3f}")
ax1.set_title("Adaptive r Distribution")
ax1.set_xlabel("r (recompute ratio)")
ax1.set_ylabel("Count")
ax1.legend(fontsize=9)

# Plot 2: adaptive_r vs TTFT scatter — does lower r → lower TTFT?
ax2 = axes[1]
sc = ax2.scatter(hq_df["ada_r_used"], hq_df["ada_ttft_ms"],
                 c=hq_df["ada_rougeL"], cmap="YlGn", s=60, alpha=0.8, edgecolors="black", linewidth=0.5)
plt.colorbar(sc, ax=ax2, label="ROUGE-L")
ax2.axvline(HQ_TARGET_RATIO, color="green", linestyle="--", linewidth=1.5, label=f"Fixed r={HQ_TARGET_RATIO}")
ax2.set_title("Adaptive r vs TTFT\n(color = ROUGE-L)")
ax2.set_xlabel("Adaptive r chosen")
ax2.set_ylabel("TTFT (ms)")
ax2.legend(fontsize=9)

# Plot 3: paired TTFT — hkvd vs adaptive per sample
ax3 = axes[2]
lims = [min(hq_df["hkvd_ttft_ms"].min(), hq_df["ada_ttft_ms"].min()) * 0.9,
        max(hq_df["hkvd_ttft_ms"].max(), hq_df["ada_ttft_ms"].max()) * 1.1]
ax3.scatter(hq_df["hkvd_ttft_ms"], hq_df["ada_ttft_ms"], alpha=0.6, s=50, color="#ffc000", edgecolors="black", linewidth=0.5)
ax3.plot(lims, lims, "k--", linewidth=1, label="Equal latency")
ax3.set_title("Per-Sample TTFT: hkvd vs adaptive")
ax3.set_xlabel("hkvd TTFT (ms)")
ax3.set_ylabel("adaptive TTFT (ms)")
ax3.set_xlim(lims); ax3.set_ylim(lims)
ax3.legend(fontsize=9)

# Points below diagonal = adaptive faster than hkvd (lower r chosen)
n_faster = int((hq_df["ada_ttft_ms"] < hq_df["hkvd_ttft_ms"]).sum())
ax3.set_title(f"TTFT: hkvd vs adaptive\n({n_faster}/{len(hq_df)} samples adaptive faster)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/hotpotqa_extension_c_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULTS_DIR}/hotpotqa_extension_c_analysis.png")

print(f"\nExtension C summary:")
print(f"  Fixed r (hkvd)          : {hq_df['hkvd_r_used'].mean():.4f} ± {hq_df['hkvd_r_used'].std():.4f}")
print(f"  Adaptive r mean         : {hq_df['ada_r_used'].mean():.4f} ± {hq_df['ada_r_used'].std():.4f}")
print(f"  Samples where ada < hkvd r: {(hq_df['ada_r_used'] < hq_df['hkvd_r_used']).sum()}/{len(hq_df)}")
print(f"  Samples where adaptive faster: {n_faster}/{len(hq_df)}")

In [ ]:
# 14c) HotpotQA visualizations — 4-way comparison
sns.set_theme(style="whitegrid", context="talk", palette="muted")

# ── Data ──────────────────────────────────────────────────────────────────────
ttft_bar = pd.DataFrame({
    "Mode"          : ["cold", "warm_standard", "warm_cb_hkvd", "warm_cb_adaptive"],
    "Mean TTFT (ms)": [hq_df["cold_ttft_ms"].mean(), hq_df["std_ttft_ms"].mean(),
                       hq_df["hkvd_ttft_ms"].mean(), hq_df["ada_ttft_ms"].mean()],
    "Std TTFT (ms)" : [hq_df["cold_ttft_ms"].std(),  hq_df["std_ttft_ms"].std(),
                       hq_df["hkvd_ttft_ms"].std(),  hq_df["ada_ttft_ms"].std()],
})

speedup_bar = pd.DataFrame({
    "Mode"                 : ["warm_standard", "warm_cb_hkvd", "warm_cb_adaptive"],
    "Mean Speedup vs Cold" : [hq_df["std_speedup"].mean(), hq_df["hkvd_speedup"].mean(), hq_df["ada_speedup"].mean()],
    "Std Speedup"          : [hq_df["std_speedup"].std(),  hq_df["hkvd_speedup"].std(),  hq_df["ada_speedup"].std()],
})

rouge_bar = pd.DataFrame({
    "Mode"        : ["warm_standard", "warm_cb_hkvd", "warm_cb_adaptive"],
    "Mean ROUGE-L": [hq_df["std_rougeL"].mean(), hq_df["hkvd_rougeL"].mean(), hq_df["ada_rougeL"].mean()],
    "Std ROUGE-L" : [hq_df["std_rougeL"].std(),  hq_df["hkvd_rougeL"].std(),  hq_df["ada_rougeL"].std()],
})

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("HotpotQA 4-Way Benchmark — CacheBlend vs Baselines", fontsize=16, y=1.02)

BAR_COLORS_4 = ["#e07b54", "#5b9bd5", "#70ad47", "#ffc000"]
BAR_COLORS_3 = ["#5b9bd5", "#70ad47", "#ffc000"]

# ── Plot 1: TTFT bars ─────────────────────────────────────────────────────────
ax1 = axes[0, 0]
bars = ax1.bar(ttft_bar["Mode"], ttft_bar["Mean TTFT (ms)"],
               yerr=ttft_bar["Std TTFT (ms)"], capsize=5,
               color=BAR_COLORS_4, alpha=0.85, edgecolor="black", linewidth=0.8)
ax1.set_title("Mean TTFT by Mode")
ax1.set_ylabel("TTFT (ms)")
ax1.tick_params(axis="x", rotation=15)
for bar, val in zip(bars, ttft_bar["Mean TTFT (ms)"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{val:.0f}", ha="center", va="bottom", fontsize=10)

cold_val = ttft_bar.loc[ttft_bar["Mode"]=="cold", "Mean TTFT (ms)"].values[0]
hkvd_val = ttft_bar.loc[ttft_bar["Mode"]=="warm_cb_hkvd", "Mean TTFT (ms)"].values[0]
if hkvd_val > 0:
    ax1.annotate(f"{cold_val/hkvd_val:.2f}× speedup\n(hkvd vs cold)",
                 xy=(2, hkvd_val), xytext=(2, cold_val * 0.72),
                 arrowprops=dict(arrowstyle="->", color="darkred"),
                 fontsize=10, color="darkred", ha="center")

# ── Plot 2: Speedup bars ──────────────────────────────────────────────────────
ax2 = axes[0, 1]
bars2 = ax2.bar(speedup_bar["Mode"], speedup_bar["Mean Speedup vs Cold"],
                yerr=speedup_bar["Std Speedup"], capsize=5,
                color=BAR_COLORS_3, alpha=0.85, edgecolor="black", linewidth=0.8)
ax2.axhline(1.0, color="red", linestyle="--", linewidth=1, label="No speedup (1×)")
ax2.set_title("Speedup vs Cold Prefill")
ax2.set_ylabel("Speedup (×)")
ax2.tick_params(axis="x", rotation=15)
ax2.legend(fontsize=10)
for bar, val in zip(bars2, speedup_bar["Mean Speedup vs Cold"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f"{val:.2f}×", ha="center", va="bottom", fontsize=11)

# ── Plot 3: ROUGE-L bars ──────────────────────────────────────────────────────
ax3 = axes[1, 0]
bars3 = ax3.bar(rouge_bar["Mode"], rouge_bar["Mean ROUGE-L"],
                yerr=rouge_bar["Std ROUGE-L"], capsize=5,
                color=BAR_COLORS_3, alpha=0.85, edgecolor="black", linewidth=0.8)
ax3.set_title("Mean ROUGE-L vs Gold Answer")
ax3.set_ylabel("ROUGE-L")
max_r = rouge_bar["Mean ROUGE-L"].max()
ax3.set_ylim(0, min(1.0, max_r * 1.4) if max_r > 0 else 0.05)
ax3.tick_params(axis="x", rotation=15)
for bar, val in zip(bars3, rouge_bar["Mean ROUGE-L"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f"{val:.3f}", ha="center", va="bottom", fontsize=10)
delta_hkvd = hq_df["hkvd_rougeL"].mean() - hq_df["std_rougeL"].mean()
delta_ada  = hq_df["ada_rougeL"].mean()  - hq_df["hkvd_rougeL"].mean()
ax3.set_xlabel(f"Δ(hkvd−std)={delta_hkvd:+.4f}  |  Δ(ada−hkvd)={delta_ada:+.4f}")

# ── Plot 4: Per-sample quality–latency scatter ────────────────────────────────
ax4 = axes[1, 1]
ax4.scatter(hq_df["std_ttft_ms"],  hq_df["std_rougeL"],
            label="warm_standard",      alpha=0.6, s=50, color="#5b9bd5")
ax4.scatter(hq_df["hkvd_ttft_ms"], hq_df["hkvd_rougeL"],
            label="warm_cb_hkvd",       alpha=0.6, s=50, color="#70ad47",  marker="^")
ax4.scatter(hq_df["ada_ttft_ms"],  hq_df["ada_rougeL"],
            label="warm_cb_adaptive",   alpha=0.6, s=50, color="#ffc000",  marker="s")
ax4.set_title("Per-Sample Quality vs Latency")
ax4.set_xlabel("TTFT (ms)")
ax4.set_ylabel("ROUGE-L vs Gold")
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/hotpotqa_4way_figure.pdf", bbox_inches="tight")
plt.savefig(f"{RESULTS_DIR}/hotpotqa_4way_figure.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULTS_DIR}/hotpotqa_4way_figure.{{pdf,png}}")

# ── ROUGE-L heatmap ───────────────────────────────────────────────────────────
if len(hq_df) >= 5:
    plot_n = min(20, len(hq_df))
    rouge_long = pd.melt(
        hq_df[["sample_id","std_rougeL","hkvd_rougeL","ada_rougeL"]].head(plot_n),
        id_vars="sample_id",
        value_vars=["std_rougeL","hkvd_rougeL","ada_rougeL"],
        var_name="mode", value_name="rougeL",
    )
    plt.figure(figsize=(18, 4))
    pivot = rouge_long.pivot(index="mode", columns="sample_id", values="rougeL")
    sns.heatmap(pivot, annot=False, cmap="YlGnBu", vmin=0, vmax=1,
                linewidths=0.3, cbar_kws={"label": "ROUGE-L"})
    plt.title(f"Per-Sample ROUGE-L vs Gold (first {plot_n} samples)")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/hotpotqa_4way_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {RESULTS_DIR}/hotpotqa_4way_heatmap.png")

## 14d) Report Narrative — How to Fill §5 (Evaluation)

**How to read the numbers:**

| Metric | Where it goes | What to say |
|--------|---------------|-------------|
| `cold_ttft_ms` mean | §5 Table 1 baseline row | Full-prefill cost. Anchor for all speedup claims. |
| `cb_speedup` mean | §5 Table 1 + §5.1 headline | Primary latency result. Paper claims 2.2–3.3×; our implementation on Mistral 4-bit via Colab T4 typically yields **0.8–1.5×** due to (a) quantization overhead in recompute path, (b) no pipelined KV load/recompute, (c) Python-level loop latency. Frame honestly. |
| `std_speedup` mean | §5 Table 1 | Should be highest speedup — pure cache hit, zero recompute. Expected ~2–5× on Mistral 7B 4-bit. |
| `cb_rougeL` − `std_rougeL` | §5 Table 2 / §5.2 | If positive: CacheBlend preserves more cross-attention signal. If near zero: AdaptiveSelector threshold needs tuning. |

**Template sentences for §5.1 (Latency):**

> Across 30 HotpotQA validation questions, warm standard KV caching achieves a mean TTFT speedup of **X.XX×** over cold prefill. CacheBlend with 15% selective recomputation achieves **Y.YY×** speedup, trading some of the cache-hit benefit for improved output quality.

**Template sentences for §5.2 (Quality):**

> On the ROUGE-L metric against gold HotpotQA answers, warm standard caching scores **A.AAA** while CacheBlend scores **B.BBB** (Δ = **+C.CCC**). The improvement is consistent with the paper's finding that selective recomputation of high-KV-deviation tokens recovers cross-attention context lost by naive KV reuse.

**If numbers disappoint (speedup < 1.0):**

Acknowledge in §6 (Discussion) — expected causes in our implementation:
1. No KV load / recompute pipeline overlap (paper's §4.3)
2. Python loop over chunks adds ~10ms/chunk overhead
3. 4-bit quantization upcasts to fp16 for recompute → extra cast latency
4. AdaptiveSelector cosine-sim path is not paper-faithful HKVD (layer-0 only, not gradual)

These are honest limitations and strengthen the Discussion section.

**Files saved by cells 14b–14c:**
- `/content/results/hq_benchmark_results.json` — raw + summary
- `/content/results/hq_benchmark_results.csv` — flat table
- `/content/results/hotpotqa_main_figure.pdf` — use directly in Overleaf
- `/content/results/hotpotqa_main_figure.png` — quick preview
- `/content/results/hotpotqa_rouge_heatmap.png` — optional per-sample appendix

**Download from Colab:**
```python
from google.colab import files
files.download("/content/results/hq_benchmark_results.csv")
files.download("/content/results/hotpotqa_main_figure.pdf")
```